In [ ]:
import os

# Paths shown reflect the default Jupyter Docker Stacks user directory (/home/jovyan).
code_path = '/home/jovyan/code/'

# source utility functions 
file_path = os.path.join(code_path, 'utility_functions_implementing_tabpfn_generators_iclr.py')
with open(os.path.expanduser(file_path)) as file:
    exec(file.read())

# source additional utility functions 
file_path = os.path.join(code_path, 'additional_utility_functions_for_tabpfn_generators_iclr.py')
with open(os.path.expanduser(file_path)) as file:
    exec(file.read())


In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import openml
import pyarrow.feather as feather

In [ ]:
def build_all_splits(X, split_seeds, ds_name, task_id):
    """
    Create all data-splits for each dataframe X and return:
      - splits_by_task: {task_id: {"dataset_name": str,
                                   "splits": [{"split": j, "orig": df, "hold": df}, ...]}}
      - splits_long: one big DataFrame where *dataset columns are prefixed*
                     so no cross-dataset name collisions occur.
        Metadata columns: __dataset__, __task_id__, __split__, __role__
    """
    long_chunks = []
    
    # --- generate splits for this dataset ---
    ds_splits = []
    for j, seed in enumerate(split_seeds, start=1):
        X_orig, X_hold = train_test_split(X, test_size=0.5, random_state=seed)

        # remove rows with NA values 
        X_orig = X_orig.dropna().reset_index(drop=True)
        X_hold = X_hold.dropna().reset_index(drop=True)

        # store per-dataset copies in the Python structure
        ds_splits.append({"split": j, "orig": X_orig, "hold": X_hold})

        # add copies to the long-form table
        for role, df in (("orig", X_orig), ("hold", X_hold)):
            chunk = df.copy()
            # metadata columns (placed in front)
            chunk.insert(0, "__role__", role)
            chunk.insert(0, "__split__", j)
            chunk.insert(0, "__task_id__", task_id)
            chunk.insert(0, "__dataset__", ds_name)
            long_chunks.append(chunk)

    splits_long = pd.concat(long_chunks, axis=0, ignore_index=True)
    return splits_long


In [ ]:
def build_all_synthetics_miav(
    X: pd.DataFrame,
    split_seeds,
    generator_kwargs=None,
    *,
    task_id: int = 0,
    ds_name: str = "",
):
    """
    For each split seed:
      - take a 50/50 split (use the 'orig' half),
      - drop rows with NA,
      - synthesize with miav_tabpfn_generator,
      - add metadata columns (no column prefixing),
    then stack everything into one long DataFrame.

    Returns
    -------
    syn_long : pd.DataFrame
        Rows from all splits, with metadata columns first:
        [__dataset__, __task_id__, __split__, __role__] + original data columns
    failures : list[dict]
        Any split-level errors: {'task_id', 'split', 'role', 'error'}
    """
    if generator_kwargs is None:
        generator_kwargs = {}

    all_chunks = []
    failures = []

    for j, seed in tqdm(enumerate(split_seeds, start=1), desc='Data split', total=len(split_seeds)):
        try:
            X_orig, _ = train_test_split(X, test_size=0.5, random_state=seed)

            # remove rows with NA values
            X_orig = X_orig.dropna().reset_index(drop=True)
            if X_orig.empty:
                continue

            # generate synthetic copy of the original half
            X_syn = miav_tabpfn_generator(X_orig, **generator_kwargs)

            # add metadata (insert in reverse so final left-to-right order is desired)
            chunk = X_syn.copy()
            chunk.insert(0, "__role__", "syn")
            chunk.insert(0, "__split__", np.int32(j))
            chunk.insert(0, "__task_id__", np.int32(task_id))
            chunk.insert(0, "__dataset__", str(ds_name))

            all_chunks.append(chunk)

        except Exception as e:
            failures.append({
                "task_id": task_id,
                "split": j,
                "role": "syn",
                "error": repr(e),
            })

    if all_chunks:
        syn_long = pd.concat(all_chunks, axis=0, ignore_index=True)
    else:
        syn_long = pd.DataFrame(columns=["__dataset__", "__task_id__", "__split__", "__role__"])

    return syn_long, failures



def build_all_synthetics_jf(
    X: pd.DataFrame,
    split_seeds,
    generator_kwargs=None,
    *,
    task_id: int = 0,
    ds_name: str = "",
):
    """
    For each split seed:
      - take a 50/50 split (use the 'orig' half),
      - drop rows with NA,
      - synthesize with miav_tabpfn_generator,
      - add metadata columns (no column prefixing),
    then stack everything into one long DataFrame.

    Returns
    -------
    syn_long : pd.DataFrame
        Rows from all splits, with metadata columns first:
        [__dataset__, __task_id__, __split__, __role__] + original data columns
    failures : list[dict]
        Any split-level errors: {'task_id', 'split', 'role', 'error'}
    """
    if generator_kwargs is None:
        generator_kwargs = {}

    all_chunks = []
    failures = []

    for j, seed in tqdm(enumerate(split_seeds, start=1), desc='Data split', total=len(split_seeds)):
        try:
            X_orig, _ = train_test_split(X, test_size=0.5, random_state=seed)

            # remove rows with NA values
            X_orig = X_orig.dropna().reset_index(drop=True)
            if X_orig.empty:
                continue

            # generate synthetic copy of the original half
            X_syn = joint_factorization_tabpfn_generator(X_orig, **generator_kwargs)

            # add metadata (insert in reverse so final left-to-right order is desired)
            chunk = X_syn.copy()
            chunk.insert(0, "__role__", "syn")
            chunk.insert(0, "__split__", np.int32(j))
            chunk.insert(0, "__task_id__", np.int32(task_id))
            chunk.insert(0, "__dataset__", str(ds_name))

            all_chunks.append(chunk)

        except Exception as e:
            failures.append({
                "task_id": task_id,
                "split": j,
                "role": "syn",
                "error": repr(e),
            })

    if all_chunks:
        syn_long = pd.concat(all_chunks, axis=0, ignore_index=True)
    else:
        syn_long = pd.DataFrame(columns=["__dataset__", "__task_id__", "__split__", "__role__"])

    return syn_long, failures



def build_all_synthetics_fc(
    X: pd.DataFrame,
    split_seeds,
    generator_kwargs=None,
    *,
    task_id: int = 0,
    ds_name: str = "",
):
    """
    For each split seed:
      - take a 50/50 split (use the 'orig' half),
      - drop rows with NA,
      - synthesize with miav_tabpfn_generator,
      - add metadata columns (no column prefixing),
    then stack everything into one long DataFrame.

    Returns
    -------
    syn_long : pd.DataFrame
        Rows from all splits, with metadata columns first:
        [__dataset__, __task_id__, __split__, __role__] + original data columns
    failures : list[dict]
        Any split-level errors: {'task_id', 'split', 'role', 'error'}
    """
    if generator_kwargs is None:
        generator_kwargs = {}

    all_chunks = []
    failures = []

    for j, seed in tqdm(enumerate(split_seeds, start=1), desc='Data split', total=len(split_seeds)):
        try:
            X_orig, _ = train_test_split(X, test_size=0.5, random_state=seed)

            # remove rows with NA values
            X_orig = X_orig.dropna().reset_index(drop=True)
            if X_orig.empty:
                continue

            # generate synthetic copy of the original half
            X_syn = full_conditionals_tabpfn_generator(X_orig, **generator_kwargs)

            # add metadata (insert in reverse so final left-to-right order is desired)
            chunk = X_syn.copy()
            chunk.insert(0, "__role__", "syn")
            chunk.insert(0, "__split__", np.int32(j))
            chunk.insert(0, "__task_id__", np.int32(task_id))
            chunk.insert(0, "__dataset__", str(ds_name))

            all_chunks.append(chunk)

        except Exception as e:
            failures.append({
                "task_id": task_id,
                "split": j,
                "role": "syn",
                "error": repr(e),
            })

    if all_chunks:
        syn_long = pd.concat(all_chunks, axis=0, ignore_index=True)
    else:
        syn_long = pd.DataFrame(columns=["__dataset__", "__task_id__", "__split__", "__role__"])

    return syn_long, failures




In [ ]:
output_path = '/home/jovyan/baseline_comparisons/outputs/'
split_seeds = list(range(1, 11))  # 10 splits

In [ ]:
# ----------------- Abalone data --------------------------------------------------------------

from sklearn.datasets import fetch_openml

# Fetch the Abalone dataset
abalone = fetch_openml(name="abalone", version=1, as_frame=True)

# Access the data and target
X = abalone.data
y = abalone.target

X['target'] =  y # Rings


num_idx = [1, 2, 3, 4, 5, 6, 7, 8]
cat_idx = [0]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'abalone', task_id = 0)
fname1 = os.path.join(output_path, "abalone_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='abalone'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "abalone_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='abalone'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "abalone_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='abalone'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "abalone_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# -------------------- Bank marketing data ------------------------------------------------------

import openml

# bank marketing
dataset = openml.datasets.get_dataset(44126) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 7))
cat_idx = [7]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'bank', task_id = 0)
fname1 = os.path.join(output_path, "bank_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='bank'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "bank_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='bank'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "bank_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='bank'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "bank_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# ----------------- Credit dataset -------------------------------------------------------------

dataset = openml.datasets.get_dataset(44089) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 10))
cat_idx = [10]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'credit', task_id = 0)
fname1 = os.path.join(output_path, "credit_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='credit'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "credit_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='credit'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "credit_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='credit'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "credit_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# ----------------- Eye movements dataset ----------------------------------------------

dataset = openml.datasets.get_dataset(44130) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 20))
cat_idx = [20]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'eye', task_id = 0)
fname1 = os.path.join(output_path, "eye_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='eye'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "eye_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='eye'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "eye_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='eye'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "eye_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# ---------------- House_16H dataset --------------------------------------------------------

dataset = openml.datasets.get_dataset(44123) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 16))
cat_idx = [16]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'house16h', task_id = 0)
fname1 = os.path.join(output_path, "house16h_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='house16h'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "house16h_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='house16h'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "house16h_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='house16h'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "house16h_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# ---------------------- MagicTelescope data ---------------------------------------------------

dataset = openml.datasets.get_dataset(44125) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 10))
cat_idx = [10]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'magic', task_id = 0)
fname1 = os.path.join(output_path, "magic_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='magic'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "magic_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='magic'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "magic_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='magic'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "magic_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])

In [ ]:
# ---------------- Pol data ----------------------------------------------------------------

dataset = openml.datasets.get_dataset(44122) 

X, y, _, attribute_names = dataset.get_data(target=dataset.default_target_attribute)

X['target'] = y

num_idx = list(range(0, 26))
cat_idx = [26]

X = enforce_dtypes(dat = X, 
                   num_variables = num_idx, 
                   cat_variables = cat_idx)


# ------------------ Generate 10 random splits between original and holdout sets --------------
print('generating data splits')
splits_long = build_all_splits(X, split_seeds, ds_name = 'pol', task_id = 0)
fname1 = os.path.join(output_path, "pol_orig_hold_splits.feather")
feather.write_feather(splits_long, fname1)


# ------------------ Generate MIAV synthetic datasets -----------------------------------------
print('running MIAV')
syn_long_miav, failures_miav = build_all_synthetics_miav(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='pol'
)
# Save the combined table once:
fname2 = os.path.join(output_path, "pol_syn_miav.feather")
feather.write_feather(syn_long_miav, fname2)


# ------------------- Generate JF synthetic datasets -------------------------------------------
print('running JF')
syn_long_jf, failures_jf = build_all_synthetics_jf(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='pol'
)

# Save the combined table once:
fname3 = os.path.join(output_path, "pol_syn_jf.feather")
feather.write_feather(syn_long_jf, fname3)


# ------------------- Generate FC synthetic datasets -------------------------------------------
print('running FC')
syn_long_fc, failures_fc = build_all_synthetics_fc(
    X=X,
    split_seeds=split_seeds,
    task_id=0,
    ds_name='pol'
)

# Save the combined table once:
fname4 = os.path.join(output_path, "pol_syn_fc.feather")
feather.write_feather(syn_long_fc, fname4)


print(len(failures_miav), failures_miav[:1])
print(len(failures_jf), failures_jf[:1])
print(len(failures_fc), failures_fc[:1])